In [0]:
dbutils.widgets.text("catalog", "", "catalog")
dbutils.widgets.text("schema", "", "schema")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = "my_volume"

# Create catalog if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
# Create volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

DataFrame[]

In [0]:
source_file = "/Volumes/test_catalog/default/my_volume/baby_names.csv"
target_file = "/Volumes/" + catalog + "/" + schema + "/" + volume + "/baby_names.csv"

try:
    # Check if target file exists
    dbutils.fs.ls(target_file)
    print(f"File already exists: {target_file}")

except:
    print(f"File not found. Copying from source...")
    dbutils.fs.cp(source_file, target_file)
    print("Copy completed.")

File already exists: /Volumes/mycatalog_dev/default/my_volume/baby_names.csv


In [0]:
# download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
table_name = f"{catalog}.{schema}.baby_names"
# path_table = catalog + "." + schema

In [0]:
table_name = f"{catalog}.{schema}.baby_names"
df = spark.read.csv(f"{target_file}", header=True, inferSchema=True, sep=",")
df = df.withColumnRenamed("First Name", "First_Name")
# display(df)
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
# Import functions
from pyspark.sql.functions import col, current_timestamp

# Define variables used in code below
file_path = "/databricks-datasets/structured-streaming/events"
username = spark.sql("SELECT regexp_replace(session_user(), '[^a-zA-Z0-9]', '_')").first()[0]
table_name = f"{catalog}.{schema}.{username}_etl_quickstart"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoint/etl_quickstart"

# Clear out data from previous demo execution
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
dbutils.fs.rm(checkpoint_path, True)

# Configure Auto Loader to ingest JSON data to a Delta table
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(file_path)
  .select("*", col("_metadata.file_path").alias("source_file"), current_timestamp().alias("processing_time"))
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable(table_name))

In [0]:
# df = spark.read.table(table_name)
# display(df)